# Vietnamese Medical NER / Assertion / Mapping — Kaggle runner

Chạy pipeline self-host (Qwen3-8B qua Ollama + ViHealthBERT/e5) trên 100 file `.txt`.

**Trước khi chạy — bật trong panel Settings bên phải:**
1. **Accelerator = GPU T4 x2** (hoặc P100)
2. **Internet = On** (bắt buộc: tải Ollama/model + gọi RxNorm API)
3. **Add Data** → thêm CẢ 2 dataset:
   - `data-input-viettel-race` — 100 file `.txt`
   - `ICD10-clean` — file `.xlsx` (ICD-10 đã làm sạch)

Chạy tuần tự từng cell. Khi chạy full 100 file, dùng **Save & Run All (Commit)** để job chạy độc lập.

## 1. Cấu hình — chỉnh cho khớp Dataset của bạn

In [ ]:
# ==== Slug của 2 dataset đã Add Data (đổi nếu bạn đặt tên khác) ====
INPUT_SLUG = "data-input-viettel-race"   # dataset chứa 100 file .txt
ICD_SLUG   = "ICD10-clean"               # dataset chứa file .xlsx ICD-10

import os, glob

def _find_dataset_dir(slug):
    """Tìm thư mục mount của dataset, khớp không phân biệt hoa/thường
    (Kaggle thường viết thường hóa slug)."""
    p = f"/kaggle/input/{slug}"
    if os.path.isdir(p):
        return p
    for d in glob.glob("/kaggle/input/*"):
        if os.path.basename(d).lower() == slug.lower():
            return d
    return p  # trả path gốc để thông báo lỗi rõ ràng bên dưới

# --- INPUT_DIR: tự dò .txt ở gốc dataset hoặc trong thư mục con ---
_in_root = _find_dataset_dir(INPUT_SLUG)
if glob.glob(f"{_in_root}/*.txt"):
    INPUT_DIR = _in_root
else:
    _cand = glob.glob(f"{_in_root}/**/*.txt", recursive=True)
    INPUT_DIR = os.path.dirname(_cand[0]) if _cand else _in_root

# --- ICD10_PATH: tự tìm file .xlsx trong dataset ICD ---
_icd_root = _find_dataset_dir(ICD_SLUG)
_xlsx = sorted(glob.glob(f"{_icd_root}/**/*.xlsx", recursive=True))
ICD10_PATH = _xlsx[0] if _xlsx else f"{_icd_root}/ICD10_master_active.xlsx"

REPO_URL = "https://github.com/jasmine95dn/vn-medical-ner-assertion.git"
REPO_DIR = "/kaggle/working/vn-medical-ner-assertion"

_n_txt = len(glob.glob(f"{INPUT_DIR}/*.txt"))
print("INPUT_DIR :", INPUT_DIR, "| số file .txt:", _n_txt)
print("ICD10_PATH:", ICD10_PATH, "| tồn tại:", os.path.isfile(ICD10_PATH))
assert _n_txt > 0, "Không thấy .txt — kiểm tra INPUT_SLUG và đã Add Data chưa"
assert os.path.isfile(ICD10_PATH), "Không thấy .xlsx — kiểm tra ICD_SLUG và đã Add Data chưa"

## 2. Lấy code từ GitHub (clone lần đầu, các lần sau tự `git pull`)

In [ ]:
import os, subprocess
if os.path.isdir(REPO_DIR):
    print("repo đã có → git pull")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 3. Cài thư viện Python

In [ ]:
!pip install -q -r requirements.txt --break-system-packages

## 4. Cài & khởi động Ollama, tải model Qwen3-8B

Server chạy nền; lần đầu `ollama pull qwen3:8b` tải ~5GB nên hơi lâu.

In [ ]:
import subprocess, time, os

# cài Ollama nếu chưa có
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# chạy server nền
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"])
time.sleep(8)  # chờ server sẵn sàng

subprocess.run(["ollama", "pull", "qwen3:8b"], check=True)
print("Ollama sẵn sàng.")

## 5. Smoke test — 3 file, bỏ candidate (nhanh, kiểm pipeline chạy được)

In [ ]:
!python main.py --input-dir "{INPUT_DIR}" --icd10-path "{ICD10_PATH}" --limit 3 --no-candidates

In [ ]:
# xem thử vài entity đầu ra
import json, glob
for f in sorted(glob.glob("output/*.json"))[:3]:
    print("===", f, "===")
    data = json.load(open(f, encoding="utf-8"))
    print(json.dumps(data[:5], ensure_ascii=False, indent=2))

## 6. Chấm điểm nhanh trên validation set (NER + assertion)

In [ ]:
!python evaluate.py --run --save-pred output/val_pred.json

## 7. Chạy FULL 100 file (có candidate mapping ICD-10 + RxNorm)

Lần đầu sẽ tải 2 model embedding (ViHealthBERT + e5-base) từ HuggingFace.
Nên chạy bằng **Save & Run All (Commit)** để không cần giữ tab mở.

In [ ]:
!python main.py --input-dir "{INPUT_DIR}" --icd10-path "{ICD10_PATH}"

## 8. Đóng gói output để tải về

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/output", "zip", "output")
print("Đã nén →/kaggle/working/output.zip (tải ở tab Output)")
import glob
print("Số file JSON:", len(glob.glob("output/*.json")))